# Do-as-I-Do · Reconstruction — RunPod

End-to-end hand + object **reconstruction and 6-DoF pose tracking** from a single demo video,
running the [`reconstruction/`](https://github.com/malik-group/do-as-i-do) pipeline on a RunPod
**A100 80 GB** GPU pod, from a Jupyter notebook.

### RunPod prerequisites (do these in the RunPod dashboard / pod first)
1. **Pod template.** Launch an **NVIDIA A100 80 GB** pod with a CUDA 12.x PyTorch template
   (e.g. RunPod's official `PyTorch` template). The CUDA toolkit (`nvcc` at `/usr/local/cuda`) is
   used to compile a few extensions (pytorch3d, DROID-SLAM), so prefer a dev/PyTorch image over a
   bare runtime image.
2. **Network volume (recommended).** Attach a network volume at `/workspace` so the cloned repo,
   weights, video, and MANO files persist across pod restarts. Otherwise everything lives on the
   container disk and is lost when the pod stops.
3. **Upload your inputs** onto the pod (via the RunPod file browser or `scp`/`rclone`):
   - your demo **video** (e.g. `/workspace/whisking.mp4`);
   - `MANO_RIGHT.pkl` and `MANO_LEFT.pkl` (manual, license-gated download from
     https://mano.is.tue.mpg.de) into a folder, e.g. `/workspace/mano/`.
4. **HuggingFace access.** Request access to the gated repos `facebook/sam-3d-objects` and
   `facebook/sam3`. Have a token ready (https://huggingface.co/settings/tokens) — you'll paste it
   in the auth cell, or export it as `HF_TOKEN` before launching Jupyter.

### What this notebook does
Installs Miniconda, builds the pipeline's **4 conda envs** (`sam3`, `sam3d`, `hawor`, `tapnet`),
fetches weights, lets you **click the object** on the reference frame (interactive matplotlib
`ginput` — works in real Jupyter, unlike Colab), and runs `run_pipeline.sh` end-to-end with the
click substituted by your points.

Run cells top-to-bottom. **[setup]** cells run once per pod; **[run]** cells are per-video.

## 0 · Configuration  [run]

All paths are **local filesystem paths on the pod** (no Google Drive). Edit for your video.

In [ ]:
import os

# --- Your video (path on the pod) ---
VIDEO_PATH = "/workspace/pipette.mp4"  # edit me

# --- Reference frame + object + anchor hand (same args as run_pipeline.sh) ---
FRAME_N = 125   # edit me
OBJECT  = "pipette"   # edit me
ANCHOR_HAND = "right"   # "right" or "left"

# --- MANO models (folder containing MANO_RIGHT.pkl + MANO_LEFT.pkl) ---
MANO_DIR = "/workspace/mano"    # edit me

# --- Where to clone the repo (put it on the network volume for persistence) ---
REPO_DIR = "/workspace/do-as-i-do"

# Expose to subsequent %%bash cells.
for k, v in {"VIDEO_PATH": VIDEO_PATH, "FRAME_N": str(FRAME_N), "OBJECT": OBJECT,
             "ANCHOR_HAND": ANCHOR_HAND, "MANO_DIR": MANO_DIR, "REPO_DIR": REPO_DIR}.items():
    os.environ[k] = v
print("VIDEO_PATH =", VIDEO_PATH)
print("REPO_DIR   =", REPO_DIR)

## 1 · GPU & disk sanity check  [setup]

Asserts the pod has an A100 (≥ 32 GB VRAM) and enough local disk for envs + weights (~40 GB).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
VRAM_MB=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1 | tr -d ' ')
if [ "$VRAM_MB" -lt 30000 ]; then
  echo "!! This GPU has only ${VRAM_MB} MB VRAM. The pipeline needs >= 32 GB."
fi
echo "--- nvcc (needed to compile extensions) ---"
nvcc --version 2>/dev/null || ls /usr/local/cuda/bin/nvcc 2>/dev/null || echo "nvcc not found — install a CUDA dev image"
echo "--- disk ---"
df -h /workspace 2>/dev/null || df -h /

## 2 · Notebook-kernel deps + HuggingFace login  [setup]

These packages are installed into the **Jupyter kernel's Python** (not the conda envs) — needed
for the interactive click plot later. Then authenticate to HuggingFace so the gated
`facebook/sam-3d-objects` / `facebook/sam3` checkpoints can be downloaded.

In [ ]:
# Kernel-side deps for the interactive click cell.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl", "matplotlib", "opencv-python", "numpy"])
print("kernel deps installed.")

In [ ]:
import os, getpass
# Prefer an existing HF_TOKEN env var; otherwise prompt (kept out of cell history via getpass).
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your HuggingFace token (input hidden): ")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF token set (length %d)." % len(os.environ["HF_TOKEN"]))

In [ ]:
%%bash
pip install -q 'huggingface-hub[cli]<1.0'
git config --global credential.helper store
hf auth login --token "$HF_TOKEN" --add-to-git-credential

## 3 · Install Miniconda  [setup]

If the pod image already has conda at `/opt/conda`, this is a no-op; otherwise it installs
Miniconda there. Every later `%%bash` cell re-sources it (shell state does not persist between
cells).

In [ ]:
%%bash
set -e
if [ -x /opt/conda/bin/conda ]; then
  echo "conda already installed at /opt/conda"
else
  echo "Installing Miniconda..."
  cd /tmp
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
  bash miniconda.sh -bfp /opt/conda
  rm miniconda.sh
fi
source /opt/conda/etc/profile.d/conda.sh
conda --version
# Recent conda refuses env creation until the defaults channels' ToS are accepted.
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true

## 4 · Clone the repo + submodules  [setup]

`GIT_LFS_SKIP_SMUDGE=1` so the heavy weight blobs are **not** pulled by Git LFS — they come from
`setup/02_fetch_weights.sh` later.

In [ ]:
%%bash
set -e
if [ -d "$REPO_DIR/.git" ]; then
  echo "Repo already cloned at $REPO_DIR"
else
  mkdir -p "$(dirname "$REPO_DIR")"
  cd "$(dirname "$REPO_DIR")"
  GIT_LFS_SKIP_SMUDGE=1 git clone --recurse-submodules https://github.com/malik-group/do-as-i-do.git "$(basename "$REPO_DIR")"
fi
cd "$REPO_DIR"
GIT_LFS_SKIP_SMUDGE=1 git submodule update --init --recursive
echo "--- submodule pins ---"
git submodule status

## 5 · Build the 4 conda envs  [setup]

Each cell runs the repo's own `setup/01_create_envs.sh <env>`. These are slow (~5-10 min each)
and use a lot of disk; run once.

- **`sam3`** and **`tapnet`** run unchanged (their recipes already install cu128 torch, fine on A100).
- **`sam3d`** needs two env vars set first (`PIP_EXTRA_INDEX_URL` + `PIP_FIND_LINKS`) so pip can
  find the `+cu121` torch wheels and the kaolin wheel bucket — `01_create_envs.sh` omits these,
  which is why a bare run fails on `torchaudio==2.5.1+cu121`. (Mirrors `sam-3d-objects/doc/setup.md`.)
  `CUDA_HOME`/`TORCH_CUDA_ARCH_LIST` are exported so any source-built extension targets sm_80.
- **`hawor`** runs unchanged (cu117/torch 1.13 runs fine on A100 sm_80).

If a cell fails, just re-run **that** cell — `conda env create` falls back to `env update`.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/01_create_envs.sh sam3

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
SAM3D_DIR="$REPO_DIR/reconstruction/modules/sam-3d-objects"

# sam-3d-objects/requirements.txt pins torch 2.5.1 / torchvision / torchaudio with the +cu121
# local version, and kaolin comes from NVIDIA's wheel bucket. setup/01_create_envs.sh doesn't
# set these, so pip can't find torchaudio==2.5.1+cu121. Mirrors sam-3d-objects/doc/setup.md.
export PIP_EXTRA_INDEX_URL="https://pypi.ngc.nvidia.com https://download.pytorch.org/whl/cu121"
export PIP_FIND_LINKS="https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html"
export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
export TORCH_CUDA_ARCH_LIST="8.0"     # A100 (sm_80)
export FORCE_CUDA=1
bash setup/01_create_envs.sh sam3d

# --- sam3d-objects post-installs that 01_create_envs.sh / requirements.txt don't provide ---
conda activate sam3d
pip install "setuptools<81" ninja wheel packaging einops psutil

# Un-shadow sam-3d-objects' vendored notebook/ (namespace pkg) from the PyPI `notebook`.
pip uninstall -y notebook notebook_shim jupyterlab jupyter_server 2>/dev/null || true

# Compiled extensions from the [p3d]/[inference] extras (not in requirements.txt):
pip install flash-attn --no-build-isolation
pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git
pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@stable"

# Verify (notebook.inference must be imported from $SAM3D_DIR — see §13b for why).
( cd "$SAM3D_DIR" && python -c "import notebook.inference; print('notebook.inference OK')" )
python -c "import pytorch3d, flash_attn, nvdiffrast; print('sam3d compiled exts OK')"

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
RECON="$REPO_DIR/reconstruction"
HAWOR_DIR="$RECON/modules/HaWoR"
cd "$RECON"

# Replicate setup/01_create_envs.sh (mk_hawor) inline, with two fixes this pod needs:
#  (1) several source packages (pytorch3d, torch-scatter, ...) `import torch` in their setup.py,
#      so the whole requirements.txt is installed with --no-build-isolation against the torch
#      installed below — otherwise PEP 517 build isolation hides torch (ModuleNotFoundError).
#  (2) compile their CUDA kernels with the pod's toolkit, targeting A100 (sm_80).
if ! conda env list | grep -q '^hawor '; then
  conda create -y -n hawor python=3.10
fi
conda activate hawor
conda install -y -c conda-forge ffmpeg
# torch 1.13 (cu117) does an EXACT nvcc-vs-torch CUDA-version check before compiling any
# extension, and the pod's system nvcc is 12.4 -> mismatch. Install a matching CUDA 11.7
# toolkit into the env and point CUDA_HOME at it so pytorch3d / torch-scatter / DROID-SLAM build.
conda install -y -c "nvidia/label/cuda-11.7.1" cuda-toolkit

# torch (cu117 per HaWoR README; runs fine on A100 sm_80)
pip install torch==1.13.0+cu117 torchvision==0.14.0+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117
# Pin setuptools<81 BEFORE building anything: torch 1.13's cpp_extension.py does
# `from pkg_resources import packaging`, and pkg_resources was removed in setuptools>=81.
pip install "setuptools<81" ninja wheel

# Everything in requirements.txt EXCEPT mmcv (won't build on modern torch) and chumpy@git.
# Several source packages (pytorch3d, torch-scatter, ...) `import torch` in their setup.py,
# so they MUST build with --no-build-isolation against the torch installed above — otherwise
# PEP 517 build isolation hides torch -> ModuleNotFoundError. PyG's wheel index lets
# torch-scatter use a prebuilt wheel; pytorch3d still compiles (CUDA_HOME/TORCH_CUDA_ARCH_LIST set).
export CUDA_HOME="$CONDA_PREFIX"           # use the env's CUDA 11.7 nvcc, NOT the pod's 12.4
export PATH="$CUDA_HOME/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="8.0"     # A100 (sm_80)
export FORCE_CUDA=1
export PIP_FIND_LINKS="https://data.pyg.org/whl/torch-1.13.0+cu117.html"
grep -viE "mmcv==1.3.9|chumpy@" "$HAWOR_DIR/requirements.txt" \
  | pip install -r /dev/stdin --no-build-isolation

# chumpy 0.71 is git-only (PyPI max 0.70); needs numpy at build time.
pip install "chumpy@git+https://github.com/mattloper/chumpy" --no-build-isolation
pip install pytorch-lightning==2.2.4 --no-deps   # needs pkg_resources (setuptools<81, pinned above)
pip install lightning-utilities torchmetrics==1.4.0

# DROID-SLAM (lietorch + droid-backends). The fork's setup.py hardcodes compute_120
# (Blackwell) gencode flags and ignores TORCH_CUDA_ARCH_LIST -> nvcc 11.7 rejects them.
# Patch them to A100 (sm_80) before building. No dispatch.h patch needed on torch 1.13.
sed -i 's/compute_120,code=sm_120/compute_80,code=sm_80/g; s/compute_120,code=compute_120/compute_80,code=compute_80/g' \
  "$HAWOR_DIR/thirdparty/DROID-SLAM/setup.py"
( cd "$HAWOR_DIR/thirdparty/DROID-SLAM" && python setup.py install )

# torch>=2.6 loads checkpoints weights_only=True, which rejects HaWoR's omegaconf-bearing
# ckpts. Restore the pre-2.6 default for this env.
conda env config vars set TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 -n hawor
echo "=== hawor env ready ==="
python -c "import torch,pytorch3d; print('torch',torch.__version__,'cuda',torch.version.cuda)"

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/01_create_envs.sh tapnet

## 6 · `sam3d` post-install fixes  [setup]

Two manual steps the README calls out after building the `sam3d` env
([`env/README.md`](https://github.com/malik-group/do-as-i-do/blob/main/reconstruction/env/README.md)):

1. `pip uninstall -y notebook` so `notebook.inference` (vendored inside sam-3d-objects) imports.
2. Build the Mip-Splatting `diff_gaussian_rasterization` (the `inria` GLB/texture baking backend),
   only if it isn't already importable.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3d

# 1. un-shadow the repo's notebook/ package
pip uninstall -y notebook 2>/dev/null || true

# 2. Mip-Splatting gaussian rasterizer (only if not already importable)
python - <<'PY'
import importlib.util as u
if u.find_spec("diff_gaussian_rasterization") is None:
    import subprocess, os
    work = "/workspace/mip-splatting-build"
    if not os.path.isdir(work):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/autonomousvision/mip-splatting.git", work], check=True)
    sub = os.path.join(work, "submodules", "diff-gaussian-rasterization")
    env = os.environ.copy()
    env["CUDA_HOME"] = os.environ.get("CUDA_HOME") or "/usr/local/cuda"
    env["TORCH_CUDA_ARCH_LIST"] = "8.0"     # A100 (sm_80)
    env["FORCE_CUDA"] = "1"
    subprocess.run(["python", "setup.py", "install"], cwd=sub, env=env, check=True)
    print("diff_gaussian_rasterization built.")
else:
    print("diff_gaussian_rasterization already installed; skipping.")
PY

## 7 · Fetch model weights  [setup]

Runs the repo's `setup/02_fetch_weights.sh --download`. Pulls the **SAM3D** checkpoint set from
HuggingFace (gated), the **HaWoR** / **Metric3D** / **DROID-SLAM** checkpoints, and the
**BootsTAPIR** checkpoint, then symlinks the shared heavy SAM3D files into both module dirs.

The Stage-1 **SAM3** model is auto-downloaded by `run_sam3_video.py` at runtime (also gated).

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
# 02_fetch_weights.sh uses rsync to copy the small config yamls; the pod image lacks it.
if ! command -v rsync >/dev/null 2>&1; then
  apt-get update -qq && apt-get install -y -qq rsync wget
fi
bash setup/02_fetch_weights.sh --download

## 8 · Place MANO hand models  [setup]

MANO is license-gated and cannot be auto-downloaded. Copy the two `.pkl` files you downloaded
from https://mano.is.tue.mpg.de (uploaded to the pod at `MANO_DIR`) into the paths HaWoR expects.

In [ ]:
import os, shutil, sys

HAWOR = os.path.join(os.environ["REPO_DIR"], "reconstruction/modules/HaWoR")
targets = {
    "MANO_RIGHT.pkl": f"{HAWOR}/_DATA/data/mano/MANO_RIGHT.pkl",
    "MANO_LEFT.pkl":  f"{HAWOR}/_DATA/data_left/mano_left/MANO_LEFT.pkl",
}
missing = []
for name, dst in targets.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    src = os.path.join(os.environ["MANO_DIR"], name)
    if os.path.isfile(src):
        shutil.copy2(src, dst); print(f"placed {name} -> {dst}")
    elif os.path.isfile(dst):
        print(f"{name} already present at {dst}")
    else:
        missing.append(src)
if missing:
    print("!! Could not find these MANO files (expected in MANO_DIR):")
    for m in missing: print("   ", m)
    sys.exit(1)
print("MANO models in place.")

## 9 · Setup sanity check  [setup]

Confirms all 4 envs exist and the key weights are where the pipeline expects them.

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
echo "=== conda envs ==="
conda env list
echo; echo "=== torch in each env ==="
for e in sam3 sam3d hawor tapnet; do
  echo -n "$e: "
  conda run -n "$e" python -c "import torch; print(torch.__version__, 'cuda=' + str(torch.version.cuda), 'avail=' + str(torch.cuda.is_available()))" 2>/dev/null || echo "(torch import failed)"
done
echo; echo "=== weight presence ==="
R="$REPO_DIR/reconstruction"
ls -lh "$R/weights/tapnet/bootstapir_checkpoint_v2.pt" 2>/dev/null || echo "MISSING: tapnet ckpt"
ls -lh "$R/weights/sam3d_shared/hf/" 2>/dev/null | head
ls -lh "$R/modules/HaWoR/weights/hawor/checkpoints/hawor.ckpt" 2>/dev/null || echo "MISSING: hawor.ckpt"
ls -lh "$R/modules/HaWoR/_DATA/data/mano/MANO_RIGHT.pkl" 2>/dev/null || echo "MISSING: MANO_RIGHT.pkl"
echo; echo "Setup complete if nothing above says MISSING / failed."
echo "Then continue to the Run section."

---
# Run phase (per-video)

Cells below are **[run]** — re-run them whenever you change `VIDEO_PATH` / `FRAME_N` / etc.

## 10 · Locate the video  [run]

The video already lives on the pod (no Drive to copy from). The pipeline writes its outputs
*next to* the video, so make sure `VIDEO_PATH` is on writable storage (network volume or
container disk).

In [ ]:
import os
VIDEO_PATH = os.environ["VIDEO_PATH"]
assert os.path.isfile(VIDEO_PATH), f"video not found: {VIDEO_PATH}"
VIDEO_DIR  = os.path.dirname(VIDEO_PATH)
os.environ["VIDEO_LOCAL"] = VIDEO_PATH
os.environ["VIDEO_DIR"]   = VIDEO_DIR
print("video:", VIDEO_PATH)
print("outputs will be written under:", VIDEO_DIR)

## 11 · Extract the reference frame for clicking  [run]

Uses the exact same `ffmpeg` invocation `run_pipeline.sh` will use in Step 0, so the frame
numbering matches the tracking output.

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3   # provides ffmpeg (same as run_pipeline.sh Step 0)
mkdir -p "$VIDEO_DIR/all_frames"
echo "VIDEO_LOCAL=$VIDEO_LOCAL  FRAME_N=$FRAME_N  VIDEO_DIR=$VIDEO_DIR"
# Step 0 (same as run_pipeline.sh): extract every frame, numbering from 0.
# -nostdin + </dev/null: CRITICAL in a %%bash cell — ffmpeg otherwise reads the cell's
# stdin and eats the rest of this script (the source of the earlier phantom 'll'/'l' errors).
# -fps_mode passthrough is the non-deprecated equivalent of -vsync 0 (ffmpeg 8.x warns on it).
ffmpeg -y -nostdin -i "$VIDEO_LOCAL" -fps_mode passthrough -start_number 0 \
    "$VIDEO_DIR/all_frames/%06d.png" </dev/null
# Reference frame = frame N copied to the 4-digit name run_pipeline.sh expects.
REF="$VIDEO_DIR/$(printf '%04d.png' "$FRAME_N")"
cp "$VIDEO_DIR/all_frames/$(printf '%06d.png' "$FRAME_N")" "$REF"
echo "reference frame: $REF"
ls -la "$REF"

## 12 · Click the object on the reference frame  [run]

Click 1-3 points on the object directly on the figure (a green ring marks each click; the title
shows the count). This uses matplotlib's **non-blocking** `mpl_connect('button_press_event', ...)`
hook with the `widget` (ipympl) backend. We deliberately avoid `plt.ginput()` — it blocks the
kernel, and with ipympl the figure only renders *after* the block returns, so you can never click.

Requires `ipympl` (installed in §2). If the figure is blank/non-interactive, do one page reload
(ipympl registers its frontend on load); kernel state is preserved.

In [ ]:
%matplotlib widget
import os, cv2
import matplotlib.pyplot as plt

FRAME_N = int(os.environ["FRAME_N"])
REF_PNG = os.path.join(os.environ["VIDEO_DIR"], f"{FRAME_N:04d}.png")
img = cv2.cvtColor(cv2.imread(REF_PNG), cv2.COLOR_BGR2RGB)
assert img is not None, f"could not read {REF_PNG}"
H, W = img.shape[:2]

clicks = []
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img); ax.axis('off')
ax.set_title(f"Click 1-3 points on '{os.environ['OBJECT']}' (image {W}x{H}).")

def _on_click(event):
    # event.xdata/ydata are None for clicks outside the axes; ignore those.
    if event.xdata is None or event.ydata is None:
        return
    x, y = int(round(event.xdata)), int(round(event.ydata))
    clicks.append((x, y))
    ax.plot(x, y, 'o', ms=14, mfc='none', mec='lime', mew=2)   # green ring marker
    ax.set_title(f"{len(clicks)} point(s): {clicks}  — run the next cell when done")
    fig.canvas.draw_idle()

# Keep the callback reference so it isn't GC'd, and so you can disconnect later if desired.
_click_cid = fig.canvas.mpl_connect('button_press_event', _on_click)
plt.show()

In [ ]:
# Format the collected clicks into the --points / --point_labels args run_sam3_video.py expects.
import os
OBJ_POINTS       = ";".join(f"{x},{y}" for x, y in clicks)
OBJ_POINT_LABELS = ";".join("1" for _ in clicks)   # all positive
print("Object points :", OBJ_POINTS)
print("Labels        :", OBJ_POINT_LABELS)
os.environ["OBJ_POINTS"] = OBJ_POINTS
os.environ["OBJ_POINT_LABELS"] = OBJ_POINT_LABELS

## 13 · Run the full pipeline  [run]

We make a one-line patch to `run_pipeline.sh`: swap the single `--click` token for a `--points`
/ `--point_labels` pair fed by the clicks you collected above. Everything else (env switching,
frame extraction, all stages 0-4) is the repo's own driver, untouched.

In [ ]:
# Patch a copy of run_pipeline.sh: swap `--click` for a --points/--point_labels pair, so Stage 1
# runs headlessly using the clicks you just collected. The trailing backslash that followed
# `--click` in the original is left in place, so line continuation still works.
import os
recon = os.path.join(os.environ["REPO_DIR"], "reconstruction")
src = open(f"{recon}/run_pipeline.sh").read()
assert src.count("--click") == 1, f"expected exactly one --click in run_pipeline.sh, found {src.count('--click')}"
repl = '--points "$OBJ_POINTS" --point_labels "$OBJ_POINT_LABELS"'
open(f"{recon}/run_pipeline_colab.sh", "w").write(src.replace("--click", repl))
print("wrote run_pipeline_colab.sh")

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
chmod +x run_pipeline_colab.sh
export OBJ_POINTS="$OBJ_POINTS"
export OBJ_POINT_LABELS="$OBJ_POINT_LABELS"
# HaWoR's headless renderer (moderngl/EGL) needs libEGL.so; the pod image lacks it.
ldconfig -p | grep -q libEGL.so || (apt-get update -qq && apt-get install -y -qq libegl1 libgl1 libgles2)
# Headless matplotlib (the §12 %matplotlib widget cell sets MPLBACKEND in the kernel env).
export MPLBACKEND=Agg
echo "=== launching patched pipeline ==="
./run_pipeline_colab.sh "$VIDEO_LOCAL" "$FRAME_N" "$OBJECT" "$ANCHOR_HAND"

## 13b · If Step 2 failed in the `sam3d` env — fix `notebook` + `pytorch3d`, then resume  [run]

Two things the `sam3d` env needs that `01_create_envs.sh` doesn't fully set up:

1. **`pip uninstall -y notebook`** — the PyPI `notebook` (Jupyter) ships a top-level `notebook/`
   module that shadows sam-3d-objects' *vendored* `notebook/` (the one with `notebook.inference`).
   Without this, `generate_mesh_sam3d.py` dies on `from notebook.inference import ...`.
2. **`pytorch3d`** — not in `requirements.txt` (it's the `[p3d]` extra); build it from source.

Run this once, then resume the pipeline from Step 2 so Steps 0-1 (frames + SAM3) aren't redone.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3d
SAM3D_DIR="$REPO_DIR/reconstruction/modules/sam-3d-objects"

# Un-shadow sam-3d-objects' vendored notebook/ (namespace pkg) from the PyPI `notebook`.
pip uninstall -y notebook notebook_shim jupyterlab jupyter_server 2>/dev/null || true

pip install "setuptools<81" ninja wheel packaging einops psutil
export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
export TORCH_CUDA_ARCH_LIST="8.0"     # A100 (sm_80)
export FORCE_CUDA=1

# Compiled extensions the sam3d inference stack needs that requirements.txt doesn't pull
# (they live in sam-3d-objects' [p3d]/[inference] extras, which 01_create_envs.sh doesn't run).
# All built --no-build-isolation because their setup.py imports torch.

# flash_attn (sparse attention backend) — prebuilt wheel for torch2.5/cu12/py3.11 if available.
pip install flash-attn --no-build-isolation

# nvdiffrast (differentiable rasterizer, used in mesh hole-filling / texture baking).
# Not on PyPI -- install from NVlabs' git (setup.py imports torch -> --no-build-isolation).
pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

# pytorch3d.
pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@stable"

# Verify notebook.inference (and its import chain) from $SAM3D_DIR.
( cd "$SAM3D_DIR" && python -c "import notebook.inference; print('notebook.inference OK')" )
python -c "import pytorch3d, flash_attn, nvdiffrast; print('pytorch3d', pytorch3d.__version__, '| flash_attn', flash_attn.__version__, '| nvdiffrast ok')"

In [ ]:
# Generate run_pipeline_step2.sh = run_pipeline.sh with Step 0 (frames) and Step 1 (SAM3) removed,
# so you can resume from Step 2 (masks -> meshes) without redoing segmentation. Step 1's outputs
# (masks/, config.json, reference frame) already exist from the run above.
import os
recon = os.path.join(os.environ["REPO_DIR"], "reconstruction")
lines = open(f"{recon}/run_pipeline.sh").read().split("\n")
def find(marker):
    for i, l in enumerate(lines):
        if marker in l:
            return i
    return -1
i0 = find("Step 0: Extract all frames")
i2 = find("Step 2: 3D reconstruction")
assert i0 > 0 and i2 > i0, "could not locate Step 0 / Step 2 markers in run_pipeline.sh"
trimmed = lines[:i0] + lines[i2:]          # drop Step 0 + Step 1 blocks
open(f"{recon}/run_pipeline_step2.sh", "w").write("\n".join(trimmed))
print(f"wrote run_pipeline_step2.sh (dropped {i2 - i0} lines: Step 0 + Step 1)")

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
chmod +x run_pipeline_step2.sh
echo "=== resuming pipeline from Step 2 (masks -> meshes) ==="
# HaWoR's headless renderer (moderngl/EGL) needs libEGL.so; the pod image lacks it.
ldconfig -p | grep -q libEGL.so || (apt-get update -qq && apt-get install -y -qq libegl1 libgl1 libgles2)
# Headless matplotlib: the §12 %matplotlib widget cell sets MPLBACKEND in the kernel env, which
# leaks into this bash cell and crashes HaWoR's older matplotlib on import.
export MPLBACKEND=Agg
# </dev/null: some stage scripts spawn ffmpeg, which would otherwise read this cell's stdin.
./run_pipeline_step2.sh "$VIDEO_LOCAL" "$FRAME_N" "$OBJECT" "$ANCHOR_HAND" </dev/null

## 13c · Resume from the Step-3 tail (skips the slow `track_object`)  [run]

`track_object` (the slow Step-3 stage) already finished — its outputs are in
`obj_tracking_out/<OBJECT>/`. This cell runs only the fast Step-3 tail (project mesh + convert
layout to camera frame) and Step 4 (optimize translation/scale), so you can see the exact error
without redoing tracking. Run this to produce `layout_camera_frame_optimized.json`.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
source config/paths.sh

VIDEO_PATH="$(realpath "$VIDEO_LOCAL")"
n="$FRAME_N"
OBJECT_ID="${OBJECT// /_}"
ANCHOR_HAND="$ANCHOR_HAND"
VIDEO_DIR="$(dirname "$VIDEO_PATH")"
VIDEO_NAME="$(basename "${VIDEO_PATH%.*}")"
MASKS_DIR="$VIDEO_DIR/video_segmentation/masks/frame_$(printf "%06d" "$n")_masks"
CV="$VIDEO_DIR/obj_tracking_out/$OBJECT_ID/combined_visualization"

export MPLBACKEND=Agg
echo "=== contents of combined_visualization ==="
ls -la "$CV" || true

# Locate all_hand_meshes.npz. optimize_translation_scale.py INFERS it as
# video_dir/video_dir.name/all_hand_meshes.npz, which is wrong when the video sits directly in
# its parent dir (e.g. /workspace/pipette.mp4 -> /workspace/workspace/...). Find the real one and
# pass it explicitly. (HaWoR writes it under video_dir/<video_name>/.)
HAND_MESHES="$VIDEO_DIR/$VIDEO_NAME/all_hand_meshes.npz"
if [ ! -f "$HAND_MESHES" ]; then
  HAND_MESHES="$(find "$VIDEO_DIR" -name all_hand_meshes.npz -print -quit)"
fi
echo "hand meshes: $HAND_MESHES"
test -f "$HAND_MESHES"

# The object .obj exists only at the frame Step 2 ran mesh reconstruction (the ORIGINAL
# FRAME_N), not necessarily the current calibration ref-frame. Locate it via find so
# --mesh and --ref-frame can differ (mesh is canonical; ref-frame is for hand calibration).
OBJ_MESH="$(find "$VIDEO_DIR/video_segmentation/masks" -name "${OBJECT_ID}.obj" -print -quit)"
echo "object mesh: $OBJ_MESH"
test -f "$OBJ_MESH"

conda activate "$ENV_SAM3D"
cd "$SCRIPTS_DIR"

# Step 3 tail: project mesh + convert layout to camera frame (both fast).
if [ ! -f "$CV/layout_camera_frame.json" ]; then
  echo "=== project mesh ==="
  python run_project_mesh_combined.py \
      --video "$VIDEO_PATH" \
      --mesh "$MASKS_DIR/$OBJECT_ID/${OBJECT_ID}.obj" \
      --json "$CV/layout.json" \
      --output-base "$CV/projected"
  echo "=== convert layout -> camera frame ==="
  python convert_layout_to_camera_frame.py \
      --input "$CV/layout.json" \
      --output "$CV/layout_camera_frame.json"
fi

# Step 4: optimize translation/scale -> layout_camera_frame_optimized.json
# (--hand-meshes explicit to override the buggy inference.)
echo "=== optimize translation/scale (Step 4) ==="
python optimize_translation_scale.py \
    --video-dir "$VIDEO_DIR" \
    --layout-json "$CV/layout_camera_frame.json" \
    --mesh "$OBJ_MESH" \
    --hand-meshes "$HAND_MESHES" \
    --anchor-hand "$ANCHOR_HAND" \
    --ref-frame "$n"
echo "=== done; checking for optimized json ==="
ls -la "$CV/layout_camera_frame_optimized.json"

## 13d · Diagnose Stage-4 "too few hand raycast hits"  [run]

Stage 4 raycasts camera rays (from the pointmap intrinsics) through the anchor-hand mask pixels at
the reference frame against the HaWoR hand mesh. 0 hits means the hand mesh isn't where the rays
point. This cell inspects the mask, the hand-mesh vertex range, and the pointmap intrinsics so we
can see which is off.

In [ ]:
import numpy as np, cv2
VD, fidx, ANCHOR = "/workspace", 98, "right"
m = cv2.imread(f"{VD}/video_segmentation/masks/frame_{fidx:06d}_masks/{ANCHOR}_hand_0.png", cv2.IMREAD_GRAYSCALE)
ys, xs = np.nonzero(m > 0)
mask_c = np.array([xs.mean(), ys.mean()])
hm = np.load(f"{VD}/pipette/all_hand_meshes.npz")
v = hm[f"{ANCHOR}_vertices"][fidx]
fx, fy, cx, cy = 908.4, 908.4, 640.0, 360.0
u = fx*v[:,0]/v[:,2] + cx; w = fy*v[:,1]/v[:,2] + cy
mesh_c = np.array([u.mean(), w.mean()])
H, W = m.shape
print(f"mask centroid (u,v): ({mask_c[0]:.0f}, {mask_c[1]:.0f})")
print(f"mesh centroid (u,v): ({mesh_c[0]:.0f}, {mesh_c[1]:.0f})")
print(f"distance: {np.linalg.norm(mask_c-mesh_c):.0f} px   (image {W}x{H})")
print(f"x-flip check -> mesh at ({W-mesh_c[0]:.0f}, {mesh_c[1]:.0f}), dist {np.linalg.norm(mask_c-np.array([W-mesh_c[0], mesh_c[1]])):.0f} px")
for fi in [0,50,98,150,200,260]:
    vv = hm[f"{ANCHOR}_vertices"][fi]
    print(f"  frame {fi:3d} mesh centroid: ({(fx*vv[:,0]/vv[:,2]+cx).mean():.0f}, {(fy*vv[:,1]/vv[:,2]+cy).mean():.0f})")

In [ ]:
import numpy as np, cv2, os, glob

VIDEO_DIR = os.environ['VIDEO_DIR']
VIDEO_NAME = os.path.basename(os.path.splitext(os.environ['VIDEO_LOCAL'])[0])
FRAME_N = int(os.environ['FRAME_N'])
ANCHOR = os.environ['ANCHOR_HAND']
fidx = FRAME_N

# 1) anchor-hand mask at the reference frame
mp = f"{VIDEO_DIR}/video_segmentation/masks/frame_{fidx:06d}_masks/{ANCHOR}_hand_0.png"
m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if os.path.exists(mp) else None
print("hand mask:", mp)
print("  exists:", m is not None)
if m is not None:
    print(f"  nonzero pixels: {int((m>0).sum())} / {m.size}  shape={m.shape}")

# 2) HaWoR hand mesh vertex range at the reference frame
hm_path = f"{VIDEO_DIR}/{VIDEO_NAME}/all_hand_meshes.npz"
hm = np.load(hm_path)
key = f"{ANCHOR}_vertices"
print("\nhand meshes:", hm_path, " keys:", [k for k in hm.files if 'vert' in k])
if key in hm:
    V = hm[key]
    print(f"  {key}: shape={V.shape}")
    v = V[min(fidx, len(V)-1)]
    print(f"  frame {fidx}: min={v.min(0)}  max={v.max(0)}  mean={v.mean(0)}")
    print(f"  -> depth (z) range: {v[:,2].min():.4f} .. {v[:,2].max():.4f}  (camera-frame should be +z, ~0.1-1.0 m)")

# 3) pointmap + intrinsics for the reference frame
cands = glob.glob(f"{VIDEO_DIR}/{fidx:04d}_pointmap.npy") + glob.glob(f"{VIDEO_DIR}/all_frames/{fidx:06d}_pointmap.npy")
print("\npointmap candidates:", cands)
for p in cands[:1]:
    pm = np.load(p)
    print(f"  {p}: shape={pm.shape}  z-range={pm[...,2].min():.4f}..{pm[...,2].max():.4f}")
    ip = p.replace('_pointmap.npy','_intrinsics')
    for ext in ('.npy','.txt'):
        if os.path.exists(ip+ext):
            ipath=ip+ext; break
    else:
        ipath=None
    print("  intrinsics file:", ipath)
    if ipath and ipath.endswith('.txt'):
        fx,fy,cx,cy = [float(x) for x in open(ipath).read().split()[:4]]
        print(f"  fx={fx:.1f} fy={fy:.1f} cx={cx:.1f} cy={cy:.1f}")
    elif ipath and ipath.endswith('.npy'):
        K=np.load(ipath); print("  K=",K.reshape(3,3))

# 4) quick raycast sanity: project hand verts to pixels with these intrinsics, see if they land in the mask
print("\n=== projection sanity ===")
if m is not None and key in hm and cands:
    H,W = m.shape
    v = V[min(fidx, len(V)-1)]
    # try to get intrinsics
    ip = cands[0].replace('_pointmap.npy','_intrinsics')
    K=None
    for ext in ('.npy','.txt'):
        if os.path.exists(ip+ext):
            if ext=='.txt':
                fx,fy,cx,cy=[float(x) for x in open(ip+ext).read().split()[:4]]; K=np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])
            else:
                K=np.load(ip).reshape(3,3)
    if K is not None:
        z = v[:,2]
        pos = z > 0
        proj = (v[pos] @ K.T)
        uu = proj[:,0]/proj[:,2]; vv=proj[:,1]/proj[:,2]
        inn = (uu>=0)&(uu<W)&(vv>=0)&(vv<H)
        in_mask = np.zeros(len(uu), dtype=bool)
        in_mask[inn] = m[vv[inn].astype(int), uu[inn].astype(int)] > 0
        print(f"  hand verts in front of camera (z>0): {pos.sum()}/{len(v)}")
        print(f"  projected into image bounds: {inn.sum()}")
        print(f"  projected into hand MASK: {in_mask.sum()}  (this is ~what the raycast needs)")

# Compare WHERE the mask is vs WHERE the mesh projects (the crux of the 0-hits).
if m is not None and key in hm:
    ys, xs = np.nonzero(m > 0)
    mask_c = np.array([xs.mean(), ys.mean()])           # (u, v) mask centroid in pixels
    v = V[min(fidx, len(V)-1)]
    if K is not None:
        proj = v @ K.T
        uu = proj[:,0]/proj[:,2]; vv = proj[:,1]/proj[:,2]
        mesh_c = np.array([uu.mean(), vv.mean()])        # (u, v) projected mesh centroid
        print(f"  mask centroid  (u,v): ({mask_c[0]:.0f}, {mask_c[1]:.0f})")
        print(f"  mesh centroid  (u,v): ({mesh_c[0]:.0f}, {mesh_c[1]:.0f})")
        print(f"  pixel distance: {np.linalg.norm(mask_c - mesh_c):.0f} px  (image is {W}x{H})")
        # mirror check: is the mesh centroid aligned with the horizontally-flipped mask centroid?
        mesh_mirror = np.array([W - mesh_c[0], mesh_c[1]])
        print(f"  if x-flipped, mesh would be at ({mesh_mirror[0]:.0f}, {mesh_mirror[1]:.0f}), dist {np.linalg.norm(mask_c - mesh_mirror):.0f} px")
        # also report a few frames around 98 to see if alignment is systematically off
        print("  -- per-frame mesh projection centroid (u,v) --")
        for fi in [0, 50, 98, 150, 200, 260]:
            vv = V[min(fi, len(V)-1)]
            p = vv @ K.T; u = p[:,0]/p[:,2]; w = p[:,1]/p[:,2]
            print(f"    frame {fi:3d}: ({u.mean():.0f}, {w.mean():.0f})")


In [ ]:
import numpy as np, cv2, os, glob
VD, ANCHOR = "/workspace", "right"
hm = np.load(f"{VD}/pipette/all_hand_meshes.npz")
V = hm[f"{ANCHOR}_vertices"]
fx, fy, cx, cy = 908.4, 908.4, 640.0, 360.0
rows = []
for fi in range(len(V)):
    mp = f"{VD}/video_segmentation/masks/frame_{fi:06d}_masks/{ANCHOR}_hand_0.png"
    if not os.path.exists(mp): continue
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    nz = int((m>0).sum())
    if nz < 500: continue
    ys, xs = np.nonzero(m>0); mask_c = np.array([xs.mean(), ys.mean()])
    v = V[fi]
    u = fx*v[:,0]/v[:,2]+cx; w = fy*v[:,1]/v[:,2]+cy
    # fraction of mesh verts projecting INSIDE the hand mask
    ui, wi = u.astype(int), w.astype(int)
    ok = (ui>=0)&(ui<1280)&(wi>=0)&(wi<720)
    inn = np.zeros(len(u), bool); inn[ok] = m[wi[ok], ui[ok]]>0
    rows.append((fi, nz, int(inn.sum()), np.hypot(*(mask_c-[u.mean(),w.mean()]))))
rows.sort(key=lambda r: -r[2])   # by # mesh verts inside mask, descending
print("best frames (fi, mask_px, mesh_verts_in_mask, centroid_dist_px):")
for r in rows[:15]: print(" ", r)

## 14 · Done — where the outputs live  [run]

All outputs are written next to the video under `$VIDEO_DIR/`. The directory consumed by the
[`retargeting/`](https://github.com/malik-group/do-as-i-do/tree/main/retargeting) pipeline is:

```
<VIDEO_DIR>/obj_tracking_out/<OBJECT>/combined_visualization/
    layout_camera_frame_optimized.json      <- final 6-DoF object pose track
    projected_*.png                          <- projected-mesh overlays
```

Also produced: per-object `.obj` meshes in `video_segmentation/masks/frame_NNNNNN_masks/<OBJECT>/`,
HaWoR `all_hand_meshes.npz`, per-frame `*_pointmap.npy` / `*_intrinsics.*`, and `gravity.json`.

In [ ]:
%%bash
VIDEO_DIR="$(dirname "$VIDEO_LOCAL")"
echo "=== tree of outputs ==="
find "$VIDEO_DIR" -maxdepth 3 -type d | sort
echo; echo "=== final layout json ==="
ls -la "$VIDEO_DIR/obj_tracking_out/$OBJECT/combined_visualization/layout_camera_frame_optimized.json" 2>/dev/null \
  && echo "OK: reconstruction finished successfully" \
  || echo "!! optimized layout json missing — check the pipeline log above."

## 15 · Interactive 3D visualization (viser)  [run]

Launches the viser web viewer over the reconstruction (object mesh + tracked pose + both hands).
It runs a server on `:8080` and **blocks** this cell — open it in a browser, then stop the cell (■)
when done. viser is web-based (websockets), so no X display is needed.

### Reaching port 8080 from your laptop
RunPod pods are not on the public internet — tunnel to the port:

**SSH local forward (most reliable):** from the pod's *Connect* panel copy the SSH command and add
`-L 8080:localhost:8080`, run it on your **local** machine, then open `http://localhost:8080`:
```
ssh -L 8080:localhost:8080 root@<pod>.proxy.runpod.net -p <port>
```

**RunPod HTTP proxy:** if your template exposes 8080 (see the pod's Connect/ports panel), click the
URL `https://<pod-id>-8080.proxy.runpod.net`. If the page loads blank, the server is bound to
127.0.0.1 — use the SSH forward instead (it reaches pod-localhost).

Note: the object `.obj` is located via `find` (it may have been reconstructed at a different frame
than the current `FRAME_N`).

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3d
cd "$REPO_DIR/reconstruction"

VIDEO_PATH="$(realpath "$VIDEO_LOCAL")"
VIDEO_DIR="$(dirname "$VIDEO_PATH")"
n="$FRAME_N"
OBJECT_ID="${OBJECT// /_}"
VIDEO_NAME="$(basename "${VIDEO_PATH%.*}")"
MASKS_DIR="$VIDEO_DIR/video_segmentation/masks/frame_$(printf "%06d" "$n")_masks"
# The object .obj may be at a different frame than the current ref-frame; locate it.
OBJ_MESH="$(find "$VIDEO_DIR/video_segmentation/masks" -name "${OBJECT_ID}.obj" -print -quit)"
LAYOUT_JSON_OPT="$VIDEO_DIR/obj_tracking_out/$OBJECT_ID/combined_visualization/layout_camera_frame_optimized.json"
HAND_MESHES_PATH="$VIDEO_DIR/$VIDEO_NAME/all_hand_meshes.npz"

MESH_SCALE="$(python3 -c "import json; d=json.load(open('$LAYOUT_JSON_OPT')); print(d['translation_scale_optimization']['mesh_scale'])")"
echo "mesh_scale=$MESH_SCALE  layout=$LAYOUT_JSON_OPT"
python scripts/visualize_3d.py \
    --frames-dir "$VIDEO_DIR/all_frames" \
    --layout-json "$LAYOUT_JSON_OPT" \
    --mesh "$OBJ_MESH" \
    --scale "$MESH_SCALE" \
    --translation-scale 1.0 \
    --hand-meshes "$HAND_MESHES_PATH" \
    --port 8080 </dev/null

## 16 · Zip & download results  [run]

Zips everything under `/workspace` (excluding the heavy, regenerable `do-as-i-do/` repo+weights and
`mip-splatting-build/`) into `/workspace/workspace.zip`, then download it to your laptop with:
```
scp -P 43896 -i ~/.ssh/id_ed25519 root@<pod-host>:/workspace/workspace.zip .
```
(use your pod's actual SSH host/port from *Connect → Start SSH*).

In [ ]:
%%bash
set -e
command -v zip >/dev/null 2>&1 || apt-get update -qq && apt-get install -y -qq zip
cd / && zip -r -q /workspace/workspace.zip workspace \
  -x "workspace/do-as-i-do/*" "workspace/mip-splatting-build/*" "workspace/.ipynb_checkpoints/*"
ls -lh /workspace/workspace.zip
echo "download with:  scp -P <port> -i <key> root@<pod-host>:/workspace/workspace.zip ."